# 03 · Universo CPS de personas y subtipo

- CPS = contrato válido + **persona natural** (incluye inscritas con NIT) + tipo SECOP *Prestación de servicios* + sin señales de otra familia.
- Subtipo por evidencia literal: *servicios profesionales* / *apoyo a la gestión*. Si el objeto trae ambas, se desempata con la
  **profesión u oficio nombrado** ("como arquitecto" → Profesional; "como técnico" → Apoyo). Lo que siga sin evidencia queda **ambiguo**, sin imputar.
- Revisión manual reproducible: `datos/referencias/03_revision_manual_cps.csv`. Mientras no se diligencie, las comparaciones por subtipo quedan marcadas `SIN_VALIDACION_MANUAL`.
- La ESE registra sus servicios como *Decreto 092*: se conservan en un universo **Por revisar** separado.

In [1]:
import hashlib, re, sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

RAIZ = Path.cwd().resolve()
# Busca la raíz del proyecto (carpeta que contiene funciones/secop_utils.py)
for _p in [RAIZ, *RAIZ.parents][:6]:
    if (_p / "funciones" / "secop_utils.py").exists():
        RAIZ = _p
        break
else:
    raise FileNotFoundError("No se encontró funciones/secop_utils.py; abra el notebook dentro del proyecto")
sys.path.insert(0, str(RAIZ / "funciones"))
import secop_utils as su

VERSION_NB = "03.v2.1"
ETAPA = "03_cps"
SALIDA = su.carpeta_etapa(RAIZ, ETAPA)
man02, r02 = su.abrir_etapa(RAIZ, "02_calidad")
b = su.leer_csv(r02["base_calidad"])
print(f"Entrada 02 verificada: {len(b):,} contratos")

Entrada 02 verificada: 37,574 contratos


## Señales de texto

In [2]:
tipo = su.normalizar_texto(b["tipo_de_contrato"]).fillna("")
modalidad = su.normalizar_texto(b["modalidad_de_contratacion"]).fillna("")
justif = su.normalizar_texto(b["justificacion_modalidad"]).fillna("")
desc = su.normalizar_texto(b["descripcion_del_proceso"]).fillna("")

P_PROF = r"\bservicios? profesionales?\b"
P_APOYO = r"\bapoyo a la gestion\b|\bservicios? de apoyo\b"
b["flag_prof_desc"] = desc.str.contains(P_PROF, regex=True)
b["flag_apoyo_desc"] = desc.str.contains(P_APOYO, regex=True)
b["flag_prof_just"] = justif.str.contains(P_PROF, regex=True)
b["flag_apoyo_just"] = justif.str.contains(P_APOYO, regex=True)
b["flag_tecnico_desc"] = desc.str.contains(r"\bcomo tecnic|\bcomo tecnolog|\bcomo auxiliar|\bservicios tecnicos\b", regex=True)
hay_desc = b["flag_prof_desc"] | b["flag_apoyo_desc"]
prof = b["flag_prof_desc"] | (~hay_desc & b["flag_prof_just"])
apoyo = b["flag_apoyo_desc"] | (~hay_desc & b["flag_apoyo_just"])
b["subtipo_cps"] = np.select([prof & ~apoyo, apoyo & ~prof, prof & apoyo],
                             ["Profesional", "Apoyo a la gestión", "Ambiguo"], default="Sin evidencia")
b["fuente_subtipo"] = np.select([hay_desc, prof | apoyo], ["Descripción", "Justificación"], default="Ninguna")

## Desempate por la profesión u oficio nombrado en el objeto
Muchos objetos dicen a la vez "servicios profesionales y de apoyo a la gestión" (subtipo literal **Ambiguo**) pero nombran el rol:
"como arquitecto", "como técnico en sistemas", "como auxiliar de enfermería". Se lee el rol en el tramo inicial del objeto (antes de
"para", "que" o ";") y gana **el primer rol que aparece**, para que "tecnólogo en ingeniería" no se lea como ingeniero.
Regla (Decreto 1082 de 2015, art. 2.2.1.2.1.4.9): profesión universitaria → **Profesional**; técnico, tecnólogo, auxiliar, formador,
monitor, instructor u oficio → **Apoyo a la gestión**. Solo se aplica a los ambiguos; la etiqueta literal se conserva en `subtipo_literal`.
La concordancia entre la etiqueta literal y el rol, donde ambas existen, sirve como validación interna del clasificador.

In [3]:
ROL_APOYO = re.compile(
    r"\b(?:tecnic[oa]s?|tecnolog[oa]s?|auxiliar(?:es)?|monitor(?:a|es)?|formador(?:a|es)?|instructor(?:a|es)?|tallerista|"
    r"promotor(?:a|es)?|operari[oa]s?|conductor(?:a|es)?|mensajer[oa]s?|secretari[oa]|digitador(?:a|es)?|"
    r"gestor(?:a|es)? (?:musical|cultural|comunitari[oa]|social|deportiv[oa])|mentor(?:a|es)?|bachiller(?:es)?|vigia|celador(?:a)?|"
    r"oficios varios|servicios tecnicos|tecnico laboral)\b")
ROL_PROF = re.compile(
    r"\b(?:ingenier[oa]s?|arquitect[oa]s?|abogad[oa]s?|contador(?:a|es)?|comunicador(?:a|es)? social(?:es)?|psicolog[oa]s?|"
    r"trabajador(?:a|es)? social(?:es)?|medic[oa]s?|odontolog[oa]s?|enfermer[oa]s? (?:profesional|jefe)|nutricionista|fisioterapeuta|"
    r"economista|administrador(?:a|es)? (?:de empresas|public[oa]|financier[oa]|ambiental|de negocios)|licenciad[oa]s?|"
    r"(?<!servicios )(?<!servicio )profesional(?:es)? (?:en|universitari[oa]|especializad[oa])|especialista|magister|zootecnista|"
    r"veterinari[oa]|geolog[oa]|biolog[oa]|quimic[oa]|antropolog[oa]|sociolog[oa]|politolog[oa]|historiador(?:a)?|bacteriolog[oa]|"
    r"fonoaudiolog[oa]|terapeuta|periodista|disenador(?:a)? grafic[oa]|internacionalista)\b")


def rol_en_objeto(texto):
    tramo = re.split(r"\bpara\b|\bcon el fin\b|\bque\b|;", texto, maxsplit=1)[0][:160]
    a, p = ROL_APOYO.search(tramo), ROL_PROF.search(tramo)
    if a and (not p or a.start() < p.start()):
        return "Apoyo a la gestión"
    return "Profesional" if p else "Sin rol"


b["rol_objeto"] = desc.map(rol_en_objeto)
b["subtipo_literal"] = b["subtipo_cps"]
resuelve = b["subtipo_literal"].eq("Ambiguo") & b["rol_objeto"].ne("Sin rol")
b.loc[resuelve, "subtipo_cps"] = b.loc[resuelve, "rol_objeto"]
b.loc[resuelve, "fuente_subtipo"] = "Profesión u oficio en el objeto"
print(f"Ambiguos resueltos por el rol: {int(resuelve.sum()):,} de {int(b['subtipo_literal'].eq('Ambiguo').sum()):,}")

FAMILIAS = {
    "esal_decreto092": (tipo + " " + modalidad + " " + justif).str.contains(r"decreto\s*092|entidad sin animo de lucro|\besal\b", regex=True),
    "convenio": (tipo + " " + modalidad + " " + justif).str.contains(r"\bconvenio\b|interadministrativo", regex=True),
    "obra": tipo.str.contains(r"\bobra\b") | desc.str.contains(r"\bcontrato de obra\b|\bejecucion de obras?\b|\bconstruccion de\b", regex=True),
    "bienes": tipo.str.contains(r"suministro|compraventa|adquisicion de bienes|enajenacion", regex=True)
              | desc.str.contains(r"\bsuministro de\b|\bcompraventa de\b|\badquisicion de (?:bienes|equipos|elementos|materiales)\b", regex=True),
    "consultoria_interventoria": tipo.str.contains(r"consultoria|interventoria", regex=True),
    "seguros": tipo.str.contains(r"\bseguros?\b", regex=True) | desc.str.contains(r"\bpolizas? de seguros?\b", regex=True),
    "arrendamiento": tipo.str.contains("arrendamiento") | desc.str.contains(r"\barrendamiento de\b", regex=True),
    "comodato_fiducia_concesion": (tipo + " " + desc).str.contains(r"comodato|encargo fiduciario|\bfiducia\b|contrato de concesion", regex=True),
}
for k, v in FAMILIAS.items():
    b[f"flag_familia_{k}"] = v
cols_fam = [f"flag_familia_{k}" for k in FAMILIAS]
b["senales_familia"] = b[cols_fam].apply(lambda f: "|".join(c.replace("flag_familia_", "") for c in cols_fam if f[c]), axis=1)

Ambiguos resueltos por el rol: 2,196 de 2,509


## Universos CPS

In [4]:
b["es_natural"] = b["naturaleza_proveedor"].eq("Persona natural")
b["es_tipo_prestacion"] = tipo.str.contains(r"\bprestacion de servicios\b", regex=True)
b["es_cps_candidato"] = b["es_valido_general"] & b["es_natural"] & b["es_tipo_prestacion"]
b["flag_conflicto_familia"] = b["es_cps_candidato"] & b[cols_fam].any(axis=1)
b["es_cps"] = b["es_cps_candidato"] & ~b["flag_conflicto_familia"]
b["es_cps_estricto"] = b["es_cps"] & b["subtipo_cps"].isin(["Profesional", "Apoyo a la gestión", "Ambiguo"])
b["es_cps_subtipo_claro"] = b["es_cps_estricto"] & b["subtipo_cps"].isin(["Profesional", "Apoyo a la gestión"])
b["apto_persona"] = b["es_cps_estricto"] & b["apto_identidad"]
b["apto_intervalo"] = b["apto_persona"] & b["duracion_dias_incl"].between(1, 366) & b["fecha_inicio"].notna()
b["apto_valor_mensual"] = b["es_cps_subtipo_claro"] & b["apto_persona"] & b["valor_contrato"].gt(0) & b["duracion_dias_incl"].ge(30)
# Servicios de personas registrados fuera del tipo SECOP (sobre todo ESE, Decreto 092): universo Por revisar.
patron_servicio = r"\bprestacion de servicios\b|" + P_PROF + "|" + P_APOYO
b["es_servicio_persona_por_revisar"] = (b["es_valido_general"] & b["es_natural"] & ~b["es_tipo_prestacion"]
                                        & desc.str.contains(patron_servicio, regex=True))

def flujo(nombre, m):
    s = b.loc[m & b["es_central"]]
    return {"etapa": nombre, "contratos_central": len(s), "personas_central": s["documento_identidad"].nunique(),
            "contratos_todas": int(m.sum())}
flujo_cps = pd.DataFrame([
    flujo("Válidos", b["es_valido_general"]),
    flujo("Persona natural", b["es_valido_general"] & b["es_natural"]),
    flujo("Candidato CPS", b["es_cps_candidato"]),
    flujo("Sin conflicto de familia", b["es_cps"]),
    flujo("CPS estricto (evidencia literal)", b["es_cps_estricto"]),
    flujo("Subtipo claro", b["es_cps_subtipo_claro"]),
    flujo("Apto persona", b["apto_persona"]),
    flujo("Apto valor mensual (>=30 días)", b["apto_valor_mensual"]),
])
flujo_cps

,etapa,contratos_central,personas_central,contratos_todas
0,Válidos,27919,9213,37559
1,Persona natural,26392,8741,35012
2,Candidato CPS,26329,8723,32860
3,Sin conflicto de familia,26182,8671,32679
4,CPS estricto (evidencia literal),26168,8668,32611
5,Subtipo claro,26032,8647,32339
6,Apto persona,26128,8663,32560
7,Apto valor mensual (>=30 días),25172,8512,31173


## Calidad del subtipo por año (Alcaldía central)
Si la proporción de ambiguos cambia entre años, las comparaciones por subtipo pueden reflejar redacción y no contratación.

In [5]:
c = b.loc[b["es_central"] & b["es_cps_estricto"]].copy()
c["anio"] = c["fecha_firma"].dt.year
subtipo_anio = pd.crosstab(c["anio"], c["subtipo_cps"])
for col in ["Profesional", "Apoyo a la gestión", "Ambiguo"]:
    subtipo_anio[col] = subtipo_anio.get(col, 0)
subtipo_anio["pct_ambiguo"] = 100 * subtipo_anio["Ambiguo"] / subtipo_anio.sum(axis=1)
subtipo_anio["pct_ambiguo_con_senal_tecnica"] = (
    100 * c.loc[c["subtipo_cps"].eq("Ambiguo")].groupby("anio")["flag_tecnico_desc"].mean())
denom = subtipo_anio[["Profesional", "Apoyo a la gestión", "Ambiguo"]].sum(axis=1)
subtipo_anio["pct_profesional_minimo"] = 100 * subtipo_anio["Profesional"] / denom
subtipo_anio["pct_profesional_maximo"] = 100 * (subtipo_anio["Profesional"] + subtipo_anio["Ambiguo"]) / denom
subtipo_anio = subtipo_anio.reset_index()
# Validación interna: donde la etiqueta literal es clara y el objeto nombra un rol, ¿coinciden?
cl = b.loc[b["es_cps_estricto"] & b["subtipo_literal"].isin(["Profesional", "Apoyo a la gestión"]) & b["rol_objeto"].ne("Sin rol")]
concordancia = (pd.crosstab(cl["subtipo_literal"], cl["rol_objeto"])
                .assign(concordancia_pct=lambda t: 100 * pd.Series({i: t.loc[i, i] for i in t.index}) / t.sum(axis=1))
                .reset_index())
resueltos_anio = (b.loc[b["es_central"] & b["es_cps_estricto"] & b["subtipo_literal"].eq("Ambiguo")]
                  .assign(anio=lambda d: d["fecha_firma"].dt.year)
                  .pipe(lambda d: pd.crosstab(d["anio"], d["subtipo_cps"]))
                  .add_prefix("literal_ambiguo_ahora_").reset_index())
subtipo_anio = subtipo_anio.merge(resueltos_anio, on="anio", how="left")
display(concordancia.round(1))
subtipo_anio.round(1)

rol_objeto,subtipo_literal,Apoyo a la gestión,Profesional,concordancia_pct
0,Apoyo a la gestión,2661,11,99.6
1,Profesional,232,12616,98.2


subtipo_cps,anio,Ambiguo,Apoyo a la gestión,Profesional,pct_ambiguo,pct_ambiguo_con_senal_tecnica,pct_profesional_minimo,pct_profesional_maximo,literal_ambiguo_ahora_Ambiguo,literal_ambiguo_ahora_Apoyo a la gestión,literal_ambiguo_ahora_Profesional
0,2021,27,1418,1397,1.0,3.7,49.2,50.1,27,167,115
1,2022,49,2972,2523,0.9,4.1,45.5,46.4,49,243,260
2,2023,25,2093,2267,0.6,0.0,51.7,52.3,25,185,231
3,2024,14,1986,2567,0.3,0.0,56.2,56.5,14,273,196
4,2025,8,2387,2253,0.2,0.0,48.5,48.6,8,145,53
5,2026,13,2263,1906,0.3,0.0,45.6,45.9,13,108,42


## Revisión manual reproducible
Si el archivo no existe se crea una muestra estratificada (año × subtipo) más todos los conflictos de familia de la Alcaldía.
Columnas a diligenciar: `decision_cps_manual` (SI/NO), `subtipo_manual` (Profesional / Apoyo a la gestión / Técnico / Otro), `evidencia_url`, `revisor`, `fecha_revision`.
La plantilla se rehace sola si quedó de una versión anterior del clasificador y aún no tiene filas diligenciadas; si ya hay revisiones, se conserva.

In [6]:
ruta_rev = RAIZ / "datos" / "referencias" / "03_revision_manual_cps.csv"
COLS_REV = ["id_contrato", "anio", "entidad", "subtipo_cps", "rol_objeto", "senales_familia", "descripcion_del_proceso", "url_secop",
            "version_clasificacion", "decision_cps_manual", "subtipo_manual", "evidencia_url", "revisor", "fecha_revision"]
# Se regenera si quedó de una versión anterior del clasificador y todavía nadie ha diligenciado filas
rehacer = True
if ruta_rev.exists():
    previo = pd.read_csv(ruta_rev, dtype=str, encoding="utf-8-sig")
    diligenciadas = previo.get("revisor", pd.Series(dtype=str)).notna().sum()
    misma_version = previo.get("version_clasificacion", pd.Series(dtype=str)).eq(VERSION_NB).all()
    rehacer = not misma_version and diligenciadas == 0
    if not misma_version and diligenciadas:
        print(f"Plantilla de una versión anterior con {int(diligenciadas)} filas diligenciadas: se conserva.")
if rehacer:
    pool = b.loc[b["es_central"] & (b["es_cps_estricto"] | b["flag_conflicto_familia"])].copy()
    pool["anio"] = pool["fecha_firma"].dt.year
    pool["orden"] = pool["id_contrato"].map(lambda x: hashlib.sha256(f"rev_v2|{x}".encode()).hexdigest())
    muestra = (pool.loc[pool["es_cps_estricto"]].sort_values("orden")
               .groupby(["anio", "subtipo_cps"], group_keys=False).head(15))
    # Todos los que siguen ambiguos entran a revisión, más los conflictos de familia
    muestra = pd.concat([muestra, pool.loc[pool["subtipo_cps"].eq("Ambiguo")], pool.loc[pool["flag_conflicto_familia"]]]).drop_duplicates("id_contrato")
    muestra["version_clasificacion"] = VERSION_NB
    for col in COLS_REV[9:]:
        muestra[col] = pd.NA
    su.guardar_csv(muestra[COLS_REV].sort_values(["anio", "subtipo_cps", "id_contrato"]), ruta_rev)
    print(f"Plantilla de revisión creada con la clasificación {VERSION_NB}:", su.rel(ruta_rev, RAIZ), f"({len(muestra):,} filas)")
rev = pd.read_csv(ruta_rev, dtype=str, encoding="utf-8-sig")
hecha = rev.loc[rev["decision_cps_manual"].notna() & rev["revisor"].notna()].copy()
if len(hecha):
    hecha["cps_ok"] = hecha["decision_cps_manual"].str.upper().str.strip().eq("SI")
    # Se compara con la clasificación vigente (la plantilla guarda la de su fecha de creación)
    hecha["subtipo_cps"] = hecha["id_contrato"].map(b.set_index("id_contrato")["subtipo_cps"]).fillna(hecha["subtipo_cps"])
    hecha["subtipo_ok"] = hecha["subtipo_manual"].str.strip().eq(hecha["subtipo_cps"])
    precision = hecha.groupby("subtipo_cps").agg(revisados=("id_contrato", "size"),
                                                 precision_cps=("cps_ok", "mean"),
                                                 precision_subtipo=("subtipo_ok", "mean")).reset_index()
else:
    precision = pd.DataFrame(columns=["subtipo_cps", "revisados", "precision_cps", "precision_subtipo"])
claros = precision.loc[precision["subtipo_cps"].isin(["Profesional", "Apoyo a la gestión"])]
ESTADO_VALIDACION = ("VALIDADA" if len(claros) == 2 and claros["revisados"].ge(60).all()
                     and claros["precision_subtipo"].ge(0.9).all()
                     else ("PARCIAL" if len(hecha) else "SIN_VALIDACION_MANUAL"))
print(f"Revisados: {len(hecha)} de {len(rev)} | estado: {ESTADO_VALIDACION}")
precision

Plantilla de revisión creada con la clasificación 03.v2.1: datos/referencias/03_revision_manual_cps.csv (463 filas)
Revisados: 0 de 463 | estado: SIN_VALIDACION_MANUAL


,subtipo_cps,revisados,precision_cps,precision_subtipo


## ESE Barrancabermeja (universo separado)

In [7]:
ese = b.loc[b["nit_entidad"].eq(su.NIT_ESE)]
diagnostico_ese = (ese.groupby(["tipo_de_contrato", "naturaleza_proveedor"], dropna=False)
                   .agg(contratos=("id_contrato", "size"), personas=("documento_identidad", "nunique"),
                        servicios_por_revisar=("es_servicio_persona_por_revisar", "sum"),
                        cps_estricto=("es_cps_estricto", "sum")).reset_index())
diagnostico_ese

,tipo_de_contrato,naturaleza_proveedor,contratos,personas,servicios_por_revisar,cps_estricto
0,Decreto 092 de 2017,Persona jurídica,323,71,0,0
1,Decreto 092 de 2017,Persona natural,1980,1054,1848,0
2,Decreto 092 de 2017,Por revisar,22,6,0,0
3,Otro,Grupo/consorcio,2,2,0,0


## Controles y cierre

In [8]:
ctl = su.Controles()
ctl.agregar("Filas preservadas", len(b), man02["conteos"]["contratos"])
ctl.agregar("CPS estricto con proveedor no natural", int((b["es_cps_estricto"] & ~b["es_natural"]).sum()), 0)
ctl.agregar("CPS estricto con conflicto de familia", int((b["es_cps_estricto"] & b["flag_conflicto_familia"]).sum()), 0)
ctl.agregar("Duplicados secundarios dentro de CPS", int((b["es_cps"] & b["flag_duplicado_secundario"]).sum()), 0)
ctl.agregar("ESE dentro de CPS estricto", int((b["nit_entidad"].eq(su.NIT_ESE) & b["es_cps_estricto"]).sum()), 0, "Importante")
ctl.agregar("Validación manual del subtipo", ESTADO_VALIDACION, "VALIDADA", "Importante")
ctl.agregar("Concordancia etiqueta literal vs rol en el objeto (mínimo %)", round(float(concordancia["concordancia_pct"].min()), 1), ">=95",
            "Alta", pasa=bool(concordancia["concordancia_pct"].min() >= 95))
rango_amb = subtipo_anio.loc[subtipo_anio["anio"].between(2021, 2026), "pct_ambiguo"]
ctl.agregar("Diferencia de % ambiguo entre años (p.p.)", round(float(rango_amb.max() - rango_amb.min()), 1), "<=3",
            "Importante", pasa=float(rango_amb.max() - rango_amb.min()) <= 3)
tabla_ctl = ctl.tabla()
display(tabla_ctl)

COLS_SALIDA = [c for c in b.columns if not c.startswith(("flag_prof_", "flag_apoyo_"))]
salidas = {
    "base_cps": su.guardar_csv(b[COLS_SALIDA], SALIDA / "base_cps.csv"),
    "flujo_cps": su.guardar_csv(flujo_cps, SALIDA / "flujo_cps.csv"),
    "subtipo_por_anio": su.guardar_csv(subtipo_anio, SALIDA / "subtipo_por_anio_central.csv"),
    "concordancia_subtipo": su.guardar_csv(concordancia, SALIDA / "concordancia_subtipo_rol.csv"),
    "precision_manual": su.guardar_csv(precision, SALIDA / "precision_revision_manual.csv"),
    "diagnostico_ese": su.guardar_csv(diagnostico_ese, SALIDA / "diagnostico_ese.csv"),
    "controles": su.guardar_csv(tabla_ctl, SALIDA / "controles_03.csv"),
    "revision_manual": ruta_rev,
}
estado = "BLOQUEADO" if ctl.bloqueos() else ("VALIDADO_CON_ALERTAS" if ctl.alertas() else "VALIDADO")
man = su.cerrar_etapa(RAIZ, ETAPA, VERSION_NB, {"02": su.huella_entrada(man02)}, salidas,
                      reglas={"cps": "Válido + persona natural + tipo Prestación de servicios + sin otra familia.",
                              "subtipo": "Literal en descripción; justificación solo si la descripción no distingue.",
                              "valor_mensual": "Valor / (días inclusivos / 30,4375); mínimo 30 días; no es salario.",
                              "ese": "Servicios Decreto 092 quedan Por revisar; no se suman a CPS."},
                      conteos={"cps_estricto": int(b["es_cps_estricto"].sum()),
                               "cps_estricto_central": int((b["es_cps_estricto"] & b["es_central"]).sum()),
                               "validacion_subtipo": ESTADO_VALIDACION,
                               "ambiguos_resueltos_por_rol": int(resuelve.sum()),
                               "ambiguos_restantes_cps": int((b["es_cps_estricto"] & b["subtipo_cps"].eq("Ambiguo")).sum()),
                               "concordancia_minima_pct": round(float(concordancia["concordancia_pct"].min()), 1)},
                      estado=estado, alertas=ctl.bloqueos() + ctl.alertas())
if ctl.bloqueos():
    raise RuntimeError(f"Etapa 03 bloqueada: {ctl.bloqueos()}")
print(man["estado"], man["alertas"])

,prueba,resultado,esperado,severidad,pasa
0,Filas preservadas,37574,37574,Crítica,True
1,CPS estricto con proveedor no natural,0,0,Crítica,True
2,CPS estricto con conflicto de familia,0,0,Crítica,True
3,Duplicados secundarios dentro de CPS,0,0,Crítica,True
4,ESE dentro de CPS estricto,0,0,Importante,True
5,Validación manual del subtipo,SIN_VALIDACION_MANUAL,VALIDADA,Importante,False
6,Concordancia etiqueta literal vs rol en el obj...,98.2,>=95,Alta,True
7,Diferencia de % ambiguo entre años (p.p.),0.8,<=3,Importante,True


VALIDADO_CON_ALERTAS ['Validación manual del subtipo']
